# QAOA ranked mask generation

This notebook runs the quantum/QAOA part of the pruning pipeline.

It loads the Ising Hamiltonian generated by `qubo_hamiltonian.py`, runs a small QAOA statevector simulation, samples bitstrings, ranks the sampled pruning masks, and saves them for real-model evaluation.

Output files:

- `qubo_outputs/qaoa_ranked_masks.csv`
- `qubo_outputs/qaoa_ranked_masks.json`

After this notebook runs, evaluate the sampled masks with:

`top_k_mask_evaluation.py`


## Step 1 — Imports and configuration

In [ ]:
from __future__ import annotations

import csv
import json
import math
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np

try:
    from scipy.optimize import minimize
except Exception:
    minimize = None

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "qubo_outputs"

HAMILTONIAN_JSON = OUTPUT_DIR / "hamiltonian_terms.json"
ENERGY_CSV = OUTPUT_DIR / "qubo_energy_check.csv"

QAOA_RANKED_CSV = OUTPUT_DIR / "qaoa_ranked_masks.csv"
QAOA_RANKED_JSON = OUTPUT_DIR / "qaoa_ranked_masks.json"

TOP_K = 5
SHOTS = 2048
P = 1
RESTARTS = 5
MAXITER = 120
SEED = 42

# Qiskit often displays bitstrings in reversed classical-bit order.
# We keep this True so bitstring[0] corresponds to candidate/qubit 0.
REVERSE_QISKIT_BITSTRINGS = True

random.seed(SEED)
np.random.seed(SEED)

print("Project directory:", PROJECT_DIR)
print("Hamiltonian JSON:", HAMILTONIAN_JSON)
print("Energy CSV:", ENERGY_CSV)


## Step 2 — Helper functions

In [ ]:
def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if value is None:
            return default
        text = str(value).strip()
        if text == "":
            return default
        return float(text)
    except Exception:
        return default


def safe_int(value: Any, default: int = 0) -> int:
    try:
        if value is None:
            return default
        text = str(value).strip()
        if text == "":
            return default
        return int(float(text))
    except Exception:
        return default


def read_json(path: Path) -> Any:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON file: {path}")
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)


def read_csv(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV file: {path}")
    with path.open("r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        return list(reader)


def write_csv(path: Path, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


## Step 3 — Load and parse the Hamiltonian

The Hamiltonian should come from:

`qubo_outputs/hamiltonian_terms.json`

It represents the QUBO after conversion to Ising form using:

\[
x_i = \frac{1-Z_i}{2}
\]


In [ ]:
def parse_pauli_label(label: str) -> Tuple[str, int | None, int | None]:
    label = str(label).strip().replace(" ", "")

    if label == "" or set(label) <= {"I"}:
        return "constant", None, None

    z_positions = [idx for idx, char in enumerate(label) if char == "Z"]

    if len(z_positions) == 1:
        return "Z", z_positions[0], None

    if len(z_positions) == 2:
        return "ZZ", z_positions[0], z_positions[1]

    raise ValueError(f"Unsupported Pauli label with more than two Z terms: {label}")


def extract_terms_from_hamiltonian_json(data: Any):
    constant = 0.0
    z_terms: Dict[int, float] = {}
    zz_terms: Dict[Tuple[int, int], float] = {}
    n_qubits = None
    raw_terms: List[Any] = []

    if isinstance(data, dict):
        n_qubits = data.get("num_qubits") or data.get("n_qubits") or data.get("n") or data.get("qubits")
        constant = safe_float(data.get("constant", data.get("offset", data.get("constant_offset", 0.0))))

        for key in ["terms", "hamiltonian_terms", "pauli_terms"]:
            if isinstance(data.get(key), list):
                raw_terms.extend(data[key])

        if isinstance(data.get("z_terms"), list):
            for item in data["z_terms"]:
                i = safe_int(item.get("i", item.get("qubit", item.get("index", 0))))
                coeff = safe_float(item.get("coeff", item.get("coefficient", item.get("value", 0.0))))
                z_terms[i] = z_terms.get(i, 0.0) + coeff

        if isinstance(data.get("zz_terms"), list):
            for item in data["zz_terms"]:
                i = safe_int(item.get("i", item.get("q0", item.get("qubit_0", 0))))
                j = safe_int(item.get("j", item.get("q1", item.get("qubit_1", 0))))
                coeff = safe_float(item.get("coeff", item.get("coefficient", item.get("value", 0.0))))
                key = tuple(sorted((i, j)))
                zz_terms[key] = zz_terms.get(key, 0.0) + coeff

    elif isinstance(data, list):
        raw_terms = data
    else:
        raise ValueError("Unsupported Hamiltonian JSON format.")

    for term in raw_terms:
        if not isinstance(term, dict):
            continue

        coeff = safe_float(term.get("coefficient", term.get("coeff", term.get("value", term.get("weight", 0.0)))))
        term_type = str(term.get("type", term.get("kind", ""))).upper()

        if term_type in {"CONST", "CONSTANT", "OFFSET"}:
            constant += coeff
            continue

        if term_type == "Z":
            i = safe_int(term.get("i", term.get("qubit", term.get("index", 0))))
            z_terms[i] = z_terms.get(i, 0.0) + coeff
            continue

        if term_type == "ZZ":
            i = safe_int(term.get("i", term.get("q0", term.get("qubit_0", 0))))
            j = safe_int(term.get("j", term.get("q1", term.get("qubit_1", 0))))
            key = tuple(sorted((i, j)))
            zz_terms[key] = zz_terms.get(key, 0.0) + coeff
            continue

        pauli_label = term.get("pauli") or term.get("pauli_label") or term.get("label") or term.get("operator") or term.get("term")

        if pauli_label is not None:
            parsed_type, i, j = parse_pauli_label(str(pauli_label))

            if parsed_type == "constant":
                constant += coeff
            elif parsed_type == "Z":
                z_terms[i] = z_terms.get(i, 0.0) + coeff
            elif parsed_type == "ZZ":
                key = tuple(sorted((i, j)))
                zz_terms[key] = zz_terms.get(key, 0.0) + coeff

    if n_qubits is None:
        max_index = -1
        if z_terms:
            max_index = max(max_index, max(z_terms.keys()))
        if zz_terms:
            max_index = max(max_index, max(max(pair) for pair in zz_terms.keys()))
        if max_index < 0:
            raise ValueError("Could not infer number of qubits from Hamiltonian.")
        n_qubits = max_index + 1

    return int(n_qubits), float(constant), z_terms, zz_terms


hamiltonian_data = read_json(HAMILTONIAN_JSON)
n_qubits, constant, z_terms, zz_terms = extract_terms_from_hamiltonian_json(hamiltonian_data)

print("n_qubits:", n_qubits)
print("constant:", constant)
print("Z terms:", len(z_terms))
print("ZZ terms:", len(zz_terms))
print("Z terms detail:", z_terms)
print("ZZ terms detail:", zz_terms)


## Step 4 — Load classical QUBO mask metadata

The QAOA circuit samples bitstrings, but we also need metadata for each bitstring:

- QUBO/Hamiltonian energy
- predicted compression
- predicted loss penalty
- pruned blocks

This comes from:

`qubo_outputs/qubo_energy_check.csv`


In [ ]:
def load_mask_metadata(energy_csv_path: Path) -> Dict[str, Dict[str, Any]]:
    rows = read_csv(energy_csv_path)
    metadata: Dict[str, Dict[str, Any]] = {}

    for row in rows:
        bitstring = str(row.get("bitstring", row.get("mask", ""))).strip()
        if bitstring == "":
            continue

        metadata[bitstring] = {
            "bitstring": bitstring,
            "energy": safe_float(row.get("energy", row.get("qubo_energy", 0.0))),
            "compression": safe_float(row.get("compression", row.get("predicted_compression", 0.0))),
            "loss_penalty": safe_float(row.get("loss_penalty", row.get("predicted_loss_penalty", 0.0))),
            "pairwise_penalty": safe_float(row.get("pairwise_penalty", 0.0)),
            "num_pruned_blocks": safe_int(row.get("num_pruned_blocks", 0)),
            "pruned_blocks": str(row.get("pruned_blocks", "")).strip(),
        }

    if not metadata:
        raise ValueError(f"No mask metadata found in {energy_csv_path}")

    return metadata


mask_metadata = load_mask_metadata(ENERGY_CSV)
print("Loaded mask metadata rows:", len(mask_metadata))

classical_best = min(mask_metadata.values(), key=lambda row: safe_float(row["energy"]))
print("Classical best mask:", classical_best["bitstring"])
print("Classical best energy:", classical_best["energy"])
print("Classical best pruned blocks:", classical_best["pruned_blocks"])


## Step 5 — Energy calculation

We use the same Ising convention as the QUBO-to-Hamiltonian mapping:

\[
x_i = \frac{1-Z_i}{2}
\]

So:

- if `x_i = 0`, candidate is kept and `Z_i = +1`
- if `x_i = 1`, candidate is pruned and `Z_i = -1`


In [ ]:
def bitstring_to_vector(bitstring: str) -> np.ndarray:
    return np.array([1 if char == "1" else 0 for char in bitstring], dtype=int)


def bitstring_to_z_values(bitstring: str) -> np.ndarray:
    x = bitstring_to_vector(bitstring)
    return 1 - 2 * x


def ising_energy(bitstring: str, constant: float, z_terms: Dict[int, float], zz_terms: Dict[Tuple[int, int], float]) -> float:
    z = bitstring_to_z_values(bitstring)
    energy = float(constant)

    for i, coeff in z_terms.items():
        energy += coeff * z[i]

    for (i, j), coeff in zz_terms.items():
        energy += coeff * z[i] * z[j]

    return float(energy)


# quick sanity check against metadata energy for the classical best mask
best_mask = classical_best["bitstring"]
computed_energy = ising_energy(best_mask, constant, z_terms, zz_terms)
print("Best mask:", best_mask)
print("Energy from metadata:", classical_best["energy"])
print("Energy recomputed from Hamiltonian:", computed_energy)
print("Difference:", computed_energy - safe_float(classical_best["energy"]))


## Step 6 — Build QAOA circuit

For QAOA we apply:

- cost layer from the Ising Hamiltonian
- mixer layer with RX rotations

For one QAOA layer, `p = 1`.


In [ ]:
def add_cost_layer(circuit: QuantumCircuit, gamma: float, z_terms: Dict[int, float], zz_terms: Dict[Tuple[int, int], float]) -> None:
    for i, coeff in z_terms.items():
        circuit.rz(2.0 * gamma * coeff, i)

    for (i, j), coeff in zz_terms.items():
        circuit.cx(i, j)
        circuit.rz(2.0 * gamma * coeff, j)
        circuit.cx(i, j)


def add_mixer_layer(circuit: QuantumCircuit, beta: float, n_qubits: int) -> None:
    for i in range(n_qubits):
        circuit.rx(2.0 * beta, i)


def build_qaoa_circuit(n_qubits: int, z_terms: Dict[int, float], zz_terms: Dict[Tuple[int, int], float], gammas: List[float], betas: List[float], measure: bool = False) -> QuantumCircuit:
    if len(gammas) != len(betas):
        raise ValueError("gammas and betas must have the same length.")

    circuit = QuantumCircuit(n_qubits, n_qubits if measure else 0)

    for qubit in range(n_qubits):
        circuit.h(qubit)

    for gamma, beta in zip(gammas, betas):
        add_cost_layer(circuit, gamma, z_terms, zz_terms)
        add_mixer_layer(circuit, beta, n_qubits)

    if measure:
        circuit.measure(range(n_qubits), range(n_qubits))

    return circuit


# Preview one unoptimized circuit
preview_circuit = build_qaoa_circuit(n_qubits, z_terms, zz_terms, gammas=[0.1], betas=[0.2], measure=False)
print(preview_circuit)
print("Circuit depth:", preview_circuit.depth())
print("Gate counts:", preview_circuit.count_ops())


## Step 7 — Optimize QAOA angles

For 8 qubits, we can use exact statevector simulation.

The optimizer searches for QAOA parameters that minimize the expected Hamiltonian energy.

In [ ]:
def project_bitstring_from_qiskit_label(label: str, reverse_bitstrings: bool = True) -> str:
    clean = str(label).replace(" ", "").strip()
    return clean[::-1] if reverse_bitstrings else clean


def expected_energy_from_statevector(params: np.ndarray, n_qubits: int, constant: float, z_terms: Dict[int, float], zz_terms: Dict[Tuple[int, int], float], p: int, reverse_bitstrings: bool = True) -> float:
    gammas = list(params[:p])
    betas = list(params[p:])

    circuit = build_qaoa_circuit(n_qubits, z_terms, zz_terms, gammas, betas, measure=False)
    state = Statevector.from_instruction(circuit)
    probabilities = state.probabilities_dict()

    expected = 0.0

    for qiskit_label, probability in probabilities.items():
        bitstring = project_bitstring_from_qiskit_label(qiskit_label, reverse_bitstrings=reverse_bitstrings)
        energy = ising_energy(bitstring, constant, z_terms, zz_terms)
        expected += probability * energy

    return float(expected)


def random_initial_params(p: int, seed: int | None = None) -> np.ndarray:
    rng = np.random.default_rng(seed)
    gammas = rng.uniform(0.0, 2.0 * math.pi, size=p)
    betas = rng.uniform(0.0, math.pi, size=p)
    return np.concatenate([gammas, betas])


def optimize_qaoa_params(n_qubits: int, constant: float, z_terms: Dict[int, float], zz_terms: Dict[Tuple[int, int], float], p: int, restarts: int, maxiter: int, seed: int, reverse_bitstrings: bool = True):
    if minimize is None:
        print("Scipy is not available. Using random initial QAOA parameters only.")
        params = random_initial_params(p, seed=seed)
        energy = expected_energy_from_statevector(params, n_qubits, constant, z_terms, zz_terms, p, reverse_bitstrings)
        return params, energy

    best_params = None
    best_energy = float("inf")

    for restart in range(restarts):
        init = random_initial_params(p, seed=seed + restart)

        def objective(theta: np.ndarray) -> float:
            return expected_energy_from_statevector(theta, n_qubits, constant, z_terms, zz_terms, p, reverse_bitstrings)

        result = minimize(
            objective,
            init,
            method="COBYLA",
            options={"maxiter": maxiter, "rhobeg": 0.5, "disp": False},
        )

        energy = float(result.fun)
        print(f"Restart {restart + 1}/{restarts}: expected energy={energy:.8f}, success={result.success}")

        if energy < best_energy:
            best_energy = energy
            best_params = np.array(result.x, dtype=float)

    return best_params, best_energy


best_params, best_expected_energy = optimize_qaoa_params(
    n_qubits=n_qubits,
    constant=constant,
    z_terms=z_terms,
    zz_terms=zz_terms,
    p=P,
    restarts=RESTARTS,
    maxiter=MAXITER,
    seed=SEED,
    reverse_bitstrings=REVERSE_QISKIT_BITSTRINGS,
)

gammas = best_params[:P].tolist()
betas = best_params[P:].tolist()

print("Best expected energy:", best_expected_energy)
print("Gammas:", gammas)
print("Betas:", betas)


## Step 8 — Sample masks from optimized QAOA state

One QAOA run can produce many measured bitstrings.

We sample the optimized statevector using `SHOTS = 2048` and count how often each mask appears.

In [ ]:
def sample_counts_from_statevector(params: np.ndarray, n_qubits: int, z_terms: Dict[int, float], zz_terms: Dict[Tuple[int, int], float], p: int, shots: int, seed: int, reverse_bitstrings: bool = True) -> Dict[str, int]:
    rng = np.random.default_rng(seed)

    gammas = list(params[:p])
    betas = list(params[p:])

    circuit = build_qaoa_circuit(n_qubits, z_terms, zz_terms, gammas, betas, measure=False)
    state = Statevector.from_instruction(circuit)
    probabilities_dict = state.probabilities_dict()

    labels = []
    probs = []

    for qiskit_label, prob in probabilities_dict.items():
        project_label = project_bitstring_from_qiskit_label(qiskit_label, reverse_bitstrings=reverse_bitstrings)
        labels.append(project_label)
        probs.append(float(prob))

    probs_array = np.array(probs, dtype=float)
    probs_array = probs_array / probs_array.sum()

    sampled_indices = rng.choice(len(labels), size=shots, replace=True, p=probs_array)

    counts: Dict[str, int] = {}

    for idx in sampled_indices:
        label = labels[int(idx)]
        counts[label] = counts.get(label, 0) + 1

    return counts


counts = sample_counts_from_statevector(
    params=best_params,
    n_qubits=n_qubits,
    z_terms=z_terms,
    zz_terms=zz_terms,
    p=P,
    shots=SHOTS,
    seed=SEED,
    reverse_bitstrings=REVERSE_QISKIT_BITSTRINGS,
)

print("Unique sampled masks:", len(counts))
print("Total shots:", sum(counts.values()))

most_common = sorted(counts.items(), key=lambda item: item[1], reverse=True)[:10]

print("Most frequent sampled masks:")
for bitstring, count in most_common:
    probability = count / SHOTS
    energy = mask_metadata.get(bitstring, {}).get("energy")
    if energy is None:
        energy = ising_energy(bitstring, constant, z_terms, zz_terms)
    print(f"mask={bitstring}, count={count}, prob={probability:.4f}, energy={safe_float(energy):.6f}")


## Step 9 — Rank sampled masks and save output

The output CSV is intentionally compatible with `top_k_mask_evaluation.py`.

That means after this notebook, you can evaluate the QAOA-sampled top masks with:

`python top_k_mask_evaluation.py --energy-csv qubo_outputs/qaoa_ranked_masks.csv --top-k 5`


In [ ]:
def build_ranked_rows(counts: Dict[str, int], mask_metadata: Dict[str, Dict[str, Any]], constant: float, z_terms: Dict[int, float], zz_terms: Dict[Tuple[int, int], float], shots: int, top_k: int) -> List[Dict[str, Any]]:
    sampled_rows: List[Dict[str, Any]] = []

    for bitstring, count in counts.items():
        probability = count / max(shots, 1)
        metadata = mask_metadata.get(bitstring, {})

        energy = metadata.get("energy")
        if energy is None:
            energy = ising_energy(bitstring, constant, z_terms, zz_terms)

        row = {
            "source": "qaoa_sampled",
            "qaoa_rank": "",
            "bitstring": bitstring,
            "mask": bitstring,
            "measurement_count": count,
            "probability": probability,
            "energy": safe_float(energy),
            "compression": safe_float(metadata.get("compression", 0.0)),
            "loss_penalty": safe_float(metadata.get("loss_penalty", 0.0)),
            "pairwise_penalty": safe_float(metadata.get("pairwise_penalty", 0.0)),
            "num_pruned_blocks": safe_int(metadata.get("num_pruned_blocks", 0)),
            "pruned_blocks": metadata.get("pruned_blocks", ""),
        }

        sampled_rows.append(row)

    sampled_rows = sorted(sampled_rows, key=lambda row: (safe_float(row["energy"]), -safe_float(row["probability"])))
    selected_rows = sampled_rows[:top_k]

    for idx, row in enumerate(selected_rows, start=1):
        row["qaoa_rank"] = idx

    return selected_rows


ranked_rows = build_ranked_rows(
    counts=counts,
    mask_metadata=mask_metadata,
    constant=constant,
    z_terms=z_terms,
    zz_terms=zz_terms,
    shots=SHOTS,
    top_k=TOP_K,
)

fieldnames = [
    "source",
    "qaoa_rank",
    "bitstring",
    "mask",
    "measurement_count",
    "probability",
    "energy",
    "compression",
    "loss_penalty",
    "pairwise_penalty",
    "num_pruned_blocks",
    "pruned_blocks",
]

write_csv(QAOA_RANKED_CSV, ranked_rows, fieldnames)

summary = {
    "n_qubits": n_qubits,
    "p": P,
    "shots": SHOTS,
    "restarts": RESTARTS,
    "maxiter": MAXITER,
    "seed": SEED,
    "reverse_qiskit_bitstrings": REVERSE_QISKIT_BITSTRINGS,
    "z_terms": len(z_terms),
    "zz_terms": len(zz_terms),
    "best_expected_energy": best_expected_energy,
    "gammas": gammas,
    "betas": betas,
    "unique_sampled_masks": len(counts),
    "counts": counts,
    "ranked_top_masks": ranked_rows,
    "note": "Rows are QAOA-sampled masks ranked by Hamiltonian/QUBO energy. The output CSV is compatible with top_k_mask_evaluation.py."
}

write_json(QAOA_RANKED_JSON, summary)

print("Saved QAOA ranked masks CSV to:", QAOA_RANKED_CSV)
print("Saved QAOA ranked masks JSON to:", QAOA_RANKED_JSON)

print("\nTop ranked sampled masks:")
for row in ranked_rows:
    print(
        f"rank={row['qaoa_rank']}, "
        f"mask={row['bitstring']}, "
        f"prob={safe_float(row['probability']):.4f}, "
        f"energy={safe_float(row['energy']):.6f}, "
        f"blocks={row['pruned_blocks']}"
    )


## Step 10 — Next step

Now evaluate the QAOA-ranked masks on the real neural network:

```bash
python top_k_mask_evaluation.py --energy-csv qubo_outputs/qaoa_ranked_masks.csv --top-k 5
```

This will apply the top 5 QAOA-sampled masks separately to a fresh baseline model and compare actual accuracy, F1, validation loss, and parameter reduction.
